In [ ]:
import pyschematron
import pandas as pd
import os

from pyschematron import validate_document

from pathlib import Path

from lxml import etree

from pyschematron import DirectModeSchematronValidatorFactory, validate_document
from pyschematron.direct_mode.schematron.ast_visitors import ResolveExtendsVisitor, ResolveAbstractPatternsVisitor, \
    PhaseSelectionVisitor
from pyschematron.direct_mode.schematron.parsers.xml.parser import SchemaParser, ParsingContext
from pyschematron.direct_mode.xml_validation.results.svrl_builder import DefaultSVRLReportBuilder
from pyschematron.direct_mode.xml_validation.validators import SimpleSchematronXMLValidator
from pyschematron.utils import load_xml_document


# the paths to the example data and Schema
schema_directory = 'schema/'
schema_file= 'w3c_pyschematron_test.sch'
xml_test_file_directory = 'test_files/'
output_path = 'output'

def demo_functional_interface(schema_file_path, xml_file_path):
    """This example uses the functional interface, the most simple method of interacting with PySchematron. """
    # print("Validating document {} against schema {}".format(xml_file_path, schema_file_path))
    result = validate_document(Path(xml_file_path), Path(schema_file_path))
    svrl = result.get_svrl()
    
    report_str = etree.tostring(svrl, pretty_print=True).decode('utf-8')
    # print(report_str)
    # print(result.is_valid())
    return result


In [10]:
schema_file_path=Path(schema_directory, schema_file)

results_df = pd.DataFrame(columns=['File', 'Validation Result', 'Messages'])
for root, dirs, files in os.walk(xml_test_file_directory):
    for filename in files:
        print(f"Validating {filename}")
        if filename.lower().endswith('.xml'):
            file_path = os.path.join(root, filename)
            result = demo_functional_interface(schema_file_path, file_path)
            svrl = result.get_svrl()
            validated = result.is_valid()
            failed_asserts = svrl.findall('.//{http://purl.oclc.org/dsdl/svrl}failed-assert')
            messages = [fa.find('{http://purl.oclc.org/dsdl/svrl}text').text for fa in failed_asserts]
            results_df.loc[len(results_df)] = {'File': filename, 'Validation Result': validated, 'Messages': messages}

print(results_df.to_string(justify='left'))
 

Validating .DS_Store
Validating coda_tocoda_1coda_passes.xml
Validating document test_files/coda_tocoda/coda_tocoda_1coda_passes.xml against schema schema/w3c_pyschematron_test.sch
Validating coda_tocoda_nocoda.xml
Validating document test_files/coda_tocoda/coda_tocoda_nocoda.xml against schema schema/w3c_pyschematron_test.sch
Validating coda_tocoda_2tocodas_1coda.xml
Validating document test_files/coda_tocoda/coda_tocoda_2tocodas_1coda.xml against schema schema/w3c_pyschematron_test.sch
Validating coda_tocoda_2codas_passes.xml
Validating document test_files/coda_tocoda/coda_tocoda_2codas_passes.xml against schema schema/w3c_pyschematron_test.sch
Validating coda_tocoda_no_tocoda.xml
Validating document test_files/coda_tocoda/coda_tocoda_no_tocoda.xml against schema schema/w3c_pyschematron_test.sch
  File                             Validation Result Messages                                                                                            
0    coda_tocoda_1coda_passes.xml   T